In [25]:
import cv2
import time
from ultralytics import YOLO
from pathlib import Path

In [26]:
pt_path = r"C:\Users\MUHAMMADAHMAD\Desktop\InternShip Week 4\uav week4\uav week4\runs\detect\train-12\weights\best.pt"

onnx_path = r"C:\Users\MUHAMMADAHMAD\Desktop\InternShip Week 4\uav week4\uav week4\runs\detect\train-12\weights\best.onnx"

In [27]:
onnx_model = YOLO(onnx_path)
print("Model Loaded Successfully!")

Model Loaded Successfully!


In [28]:
pt_model = YOLO(pt_path)
print("Model Loaded Successfully!")

Model Loaded Successfully!


In [ ]:
def main_menu():

    print("      YOLOv8 Object Detection System")


    print("\nSelect Model")
    print("1. PyTorch (.pt)")
    print("2. ONNX (.onnx)")

    model_choice = int(input("\nEnter Choice: "))

    if model_choice == 1:
        model = pt_model
        model_name = "PyTorch"

    elif model_choice == 2:
        model = onnx_model
        model_name = "ONNX"

    else:
        print("Invalid Choice")
        return

    print(f"\nSelected Model : {model_name}")

    print("\nSelect Input Source")
    print("1. Image")
    print("2. Video")
    print("3. Webcam")

    source_choice = int(input("\nEnter Choice: "))

    return model, model_name, source_choice

In [30]:
def detect_image(model, model_name):

    image_path = input("\nEnter image path: ")

    image = cv2.imread(image_path)

    if image is None:
        print("Error: Unable to load image.")
        return

    start_time = time.time()

    results = model.predict(
        source=image,
        conf=0.25,
        verbose=False
    )

    end_time = time.time()

    inference_time = end_time - start_time
    fps = 1 / inference_time

    annotated_image = results[0].plot()

    cv2.putText(
        annotated_image,
        f"{model_name} | FPS: {fps:.2f}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )

    output_path = "output_image.jpg"

    cv2.imwrite(output_path, annotated_image)

    cv2.imshow("YOLO Detection", annotated_image)

    print(f"\nOutput saved as {output_path}")
    print(f"Inference Time : {inference_time:.4f} sec")
    print(f"FPS            : {fps:.2f}")

    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [31]:
import os

def detect_video(model, model_name):

    video_path = input("\nEnter video path: ")

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Unable to open video.")
        return

    # -------- Video Properties --------
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_video = cap.get(cv2.CAP_PROP_FPS)

    # -------- Output Folder --------
    os.makedirs("output/videos", exist_ok=True)

    video_name = os.path.basename(video_path)
    name = os.path.splitext(video_name)[0]

    output_path = f"output/videos/{name}_detected.mp4"

    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps_video,
        (frame_width, frame_height)
    )
    frame_count = 0
    total_time = 0
    while True:

        ret, frame = cap.read()

        if not ret:
            break

        start_time = time.time()

        results = model.predict(
            source=frame,
            conf=0.25,
            verbose=False
        )

        end_time = time.time()

        inference_time = end_time - start_time
        frame_count += 1
        total_time += inference_time
        fps = frame_count / total_time

        annotated_frame = results[0].plot()

        cv2.putText(
            annotated_frame,
            f"{model_name} | FPS: {fps:.2f}",
            (20,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,255,0),
            2
        )

        writer.write(annotated_frame)

        cv2.imshow("YOLO Video Detection", annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    writer.release()
    cv2.destroyAllWindows()

    print(f"\nOutput video saved at:\n{output_path}")

In [32]:
def detect_webcam(model, model_name):

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Unable to access webcam.")
        return

    frame_count = 0
    total_time = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        start_time = time.time()

        results = model.predict(
            source=frame,
            conf=0.25,
            verbose=False
        )

        end_time = time.time()

        inference_time = end_time - start_time

        frame_count += 1
        total_time += inference_time

        fps = frame_count / total_time

        annotated_frame = results[0].plot()

        cv2.putText(
            annotated_frame,
            f"{model_name} | FPS: {fps:.2f}",
            (20,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,255,0),
            2
        )

        cv2.imshow("YOLO Webcam Detection", annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [39]:
model, model_name, source = main_menu()

if source == 1:
    detect_image(model, model_name)

elif source == 2:
    detect_video(model, model_name)

elif source == 3:
    detect_webcam(model, model_name)

else:
    print("Invalid Choice")

      YOLOv8 Object Detection System

Select Model
1. PyTorch (.pt)
2. ONNX (.onnx)

Selected Model : ONNX

Select Input Source
1. Image
2. Video
3. Webcam

Output saved as output_image.jpg
Inference Time : 0.5493 sec
FPS            : 1.82
